# Deteccao de Defeitos em PCBs utilizando Faster R-CNN

>&nbsp;&nbsp;&nbsp;&nbsp;Alexandre Augusto Tescaro Oliveira

## 1. Introducao e Motivacao

> **Resumo do Estudo:** Este trabalho apresenta o desenvolvimento de um sistema de Visao Computacional para deteccao automatica de defeitos em Placas de Circuito Impresso (PCBs), utilizando a arquitetura **Faster R-CNN**. Para viabilizar a execucao local e manter comparacao consistente entre modelos, o conjunto de dados COCO foi organizado no diretorio `pcb-defect-dataset-5000-coco`, preservando a distribuicao das classes de defeito.

### 1.1 Contextualizacao
No cenario da Industria 4.0, a garantia de qualidade na fabricacao de componentes eletronicos e critica para reduzir perdas, retrabalho e falhas em campo. As Placas de Circuito Impresso (PCBs) sao a base de praticamente todos os dispositivos eletronicos modernos. Com a miniaturizacao dos componentes, a inspecao visual tornou-se mais complexa e exige solucoes automaticas robustas.

### 1.2 O Problema
Tradicionalmente, a inspecao de PCBs e realizada de forma manual por operadores humanos ou por algoritmos de visao classica baseados em regras rigidas. Esses metodos apresentam limitacoes no ambiente industrial:
>* **Fadiga Humana:** A inspecao visual repetitiva aumenta a chance de erro e inconsistencias.
>* **Baixa Escalabilidade:** A inspecao manual e lenta e cria gargalos na linha de producao.
>* **Sensibilidade de Regras:** Metodos classicos falham com variacoes de iluminacao, rotacao e ruido.

### 1.3 A Solucao Proposta
Para reduzir esses problemas e automatizar o processo de inspecao, este notebook adota Deep Learning com **Faster R-CNN** (detector de duas etapas), buscando boa qualidade de localizacao de caixas em defeitos pequenos.

O objetivo e identificar e localizar seis tipos comuns de defeitos de fabricacao:
>1. **Missing Hole**
>2. **Mouse Bite**
>3. **Open Circuit**
>4. **Short**
>5. **Spur**
>6. **Spurious Copper**

Neste baseline, a categoria COCO de id 0 (`My-First-Project`) e ignorada, mantendo foco apenas nas classes de defeito.

## 2. Analise Exploratoria dos Dados (EDA)
> Link de acesso ao notebook realizado: https://colab.research.google.com/drive/1Zy-WgUTB66TsTryg1J9_lR-0EY2i_zgV?usp=sharing
> Link de acesso ao Dataset utilizado: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset

In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "Pillow": "PIL",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "torchmetrics": "torchmetrics",
    "faster-coco-eval": "faster_coco_eval",
}

missing = [pkg for pkg, module in required_packages.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [2]:
import json
import random
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

sns.set_style("whitegrid")

/Users/alehholiveira/Desktop/Projetos/pcb-defect-detection/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuracao de caminhos e experimento
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "pcb-defect-dataset-5000-coco"
RUNS_ROOT = PROJECT_ROOT / "runs" / "detect"
PROJECT_RUN_DIR = RUNS_ROOT / "tcc_pcb_defect_detection" / "fasterrcnn_mobilenet_v3_fpn"
PROJECT_RUN_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
TEST_DIR = DATASET_ROOT / "test"

TRAIN_ANN_PATH = TRAIN_DIR / "_annotations.coco.json"
VALID_ANN_PATH = VALID_DIR / "_annotations.coco.json"
TEST_ANN_PATH = TEST_DIR / "_annotations.coco.json"

for p in [TRAIN_ANN_PATH, VALID_ANN_PATH, TEST_ANN_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Arquivo de anotacao nao encontrado: {p}")

SEED = 42
EPOCHS = 20
BATCH_SIZE = 4
NUM_WORKERS = 0  # robusto para macOS/Python 3.14
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
LR_STEP_SIZE = 15
LR_GAMMA = 0.1
IGNORE_CATEGORY_ID = 0
EVAL_ON_CPU_WHEN_MPS = True
MAP_BACKEND = "faster_coco_eval"

if torch.cuda.is_available():
    TRAIN_DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    TRAIN_DEVICE = torch.device("mps")
else:
    TRAIN_DEVICE = torch.device("cpu")

if TRAIN_DEVICE.type == "mps" and EVAL_ON_CPU_WHEN_MPS:
    EVAL_DEVICE = torch.device("cpu")
else:
    EVAL_DEVICE = TRAIN_DEVICE

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

with open(TRAIN_ANN_PATH, "r", encoding="utf-8") as f:
    train_meta = json.load(f)

all_categories = sorted(train_meta["categories"], key=lambda item: item["id"])
defect_categories = [item for item in all_categories if item["id"] != IGNORE_CATEGORY_ID]

cat_id_to_label = {item["id"]: idx + 1 for idx, item in enumerate(defect_categories)}
label_to_name = {idx + 1: item["name"] for idx, item in enumerate(defect_categories)}
label_to_cat_id = {idx + 1: item["id"] for idx, item in enumerate(defect_categories)}
NUM_CLASSES = len(label_to_name) + 1  # + background

class_df = pd.DataFrame(
    [
        {"label": label, "cat_id": label_to_cat_id[label], "class_name": label_to_name[label]}
        for label in sorted(label_to_name.keys())
    ]
)

print(f"Dataset base: {DATASET_ROOT}")
print(f"Diretorio de saida: {PROJECT_RUN_DIR}")
print(f"Treino em: {TRAIN_DEVICE}")
print(f"Validacao em: {EVAL_DEVICE}")
print(f"Categorias COCO totais: {len(all_categories)}")
print(f"Categorias de defeito utilizadas: {len(defect_categories)}")
display(class_df)

Dataset base: /Users/alehholiveira/Desktop/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-dataset-5000-coco
Diretorio de saida: /Users/alehholiveira/Desktop/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/fasterrcnn_mobilenet_v3_fpn
Treino em: mps
Validacao em: cpu
Categorias COCO totais: 7
Categorias de defeito utilizadas: 6


,label,cat_id,class_name
0,1,1,missing_hole
1,2,2,mouse_bite
2,3,3,open_circuit
3,4,4,short
4,5,5,spur
5,6,6,spurious_copper


In [4]:
class CocoDetectionDataset(Dataset):
    def __init__(self, images_dir, annotations_path, cat_id_to_label, train=False):
        self.images_dir = Path(images_dir)
        self.annotations_path = Path(annotations_path)
        self.cat_id_to_label = dict(cat_id_to_label)
        self.train = train

        with open(self.annotations_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        self.images_meta = {img["id"]: img for img in coco["images"]}
        self.image_ids = sorted(self.images_meta.keys())

        self.annotations_by_image = defaultdict(list)
        skipped = 0

        for ann in coco["annotations"]:
            cat_id = ann.get("category_id")
            if cat_id not in self.cat_id_to_label:
                continue

            image_id = ann.get("image_id")
            if image_id not in self.images_meta:
                skipped += 1
                continue

            x, y, w, h = ann.get("bbox", [0, 0, 0, 0])
            if w <= 1 or h <= 1:
                skipped += 1
                continue

            width = self.images_meta[image_id]["width"]
            height = self.images_meta[image_id]["height"]

            x1 = max(0.0, float(x))
            y1 = max(0.0, float(y))
            x2 = min(float(width), float(x + w))
            y2 = min(float(height), float(y + h))

            if x2 <= x1 or y2 <= y1:
                skipped += 1
                continue

            self.annotations_by_image[image_id].append(
                {
                    "bbox": [x1, y1, x2, y2],
                    "label": int(self.cat_id_to_label[cat_id]),
                    "iscrowd": int(ann.get("iscrowd", 0)),
                }
            )

        self.skipped_annotations = skipped

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_meta = self.images_meta[image_id]
        image_path = self.images_dir / image_meta["file_name"]

        image = Image.open(image_path).convert("RGB")
        image = F.pil_to_tensor(image).float() / 255.0

        anns = self.annotations_by_image.get(image_id, [])
        if anns:
            boxes = torch.tensor([a["bbox"] for a in anns], dtype=torch.float32)
            labels = torch.tensor([a["label"] for a in anns], dtype=torch.int64)
            iscrowd = torch.tensor([a["iscrowd"] for a in anns], dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if boxes.numel() > 0 else torch.zeros((0,), dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.train and boxes.numel() > 0 and random.random() < 0.5:
            image = torch.flip(image, dims=[2])
            width = image.shape[2]
            boxes_flipped = boxes.clone()
            boxes_flipped[:, [0, 2]] = width - boxes[:, [2, 0]]
            target["boxes"] = boxes_flipped

        return image, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


train_dataset = CocoDetectionDataset(
    images_dir=TRAIN_DIR,
    annotations_path=TRAIN_ANN_PATH,
    cat_id_to_label=cat_id_to_label,
    train=True,
)

valid_dataset = CocoDetectionDataset(
    images_dir=VALID_DIR,
    annotations_path=VALID_ANN_PATH,
    cat_id_to_label=cat_id_to_label,
    train=False,
)

test_dataset = CocoDetectionDataset(
    images_dir=TEST_DIR,
    annotations_path=TEST_ANN_PATH,
    cat_id_to_label=cat_id_to_label,
    train=False,
)

pin_memory = TRAIN_DEVICE.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)

print(f"Amostras treino: {len(train_dataset)}")
print(f"Amostras validacao: {len(valid_dataset)}")
print(f"Amostras teste: {len(test_dataset)}")
print(f"Anotacoes descartadas (train): {train_dataset.skipped_annotations}")
print(f"Anotacoes descartadas (valid): {valid_dataset.skipped_annotations}")

sample_images, sample_targets = next(iter(train_loader))
print(f"Batch de sanity check: {len(sample_images)} imagens")
print(f"Primeira imagem shape: {tuple(sample_images[0].shape)}")
print(f"Primeiro target - boxes: {sample_targets[0]['boxes'].shape}, labels: {sample_targets[0]['labels'].shape}")

Amostras treino: 2956
Amostras validacao: 396
Amostras teste: 413
Anotacoes descartadas (train): 0
Anotacoes descartadas (valid): 0
Batch de sanity check: 4 imagens
Primeira imagem shape: (3, 601, 601)
Primeiro target - boxes: torch.Size([1, 4]), labels: torch.Size([1])


## 3. Treinamento Faster R-CNN
Nesta secao definimos o modelo, loop de treinamento, avaliacao com mAP/mAR e salvamento de artefatos para reproducao.

In [5]:
# Inicializa o modelo com tentativa de pesos pre-treinados e fallback offline em caso de SSL
import os
import ssl

# Tenta normalizar CA bundle no macOS/Python.org para permitir download HTTPS dos pesos
try:
    import certifi
    ca_bundle = certifi.where()
    os.environ["SSL_CERT_FILE"] = ca_bundle
    os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=ca_bundle)
except Exception:
    ca_bundle = None

weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT

model_init_mode = "pretrained"
ssl_error_message = None

try:
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights)
except Exception as exc:
    err_txt = str(exc).lower()
    ssl_tokens = [
        "certificate_verify_failed",
        "unable to get local issuer certificate",
        "ssl",
    ]

    if any(token in err_txt for token in ssl_tokens):
        ssl_error_message = str(exc)
        print("Aviso: falha SSL ao baixar pesos pre-treinados. Fallback para inicializacao offline (sem pesos).")
        print("Dica macOS/Python.org: execute 'Install Certificates.command' para normalizar certificados.")
        model_init_mode = "from_scratch_offline"
        model = fasterrcnn_mobilenet_v3_large_fpn(weights=None, weights_backbone=None)
    else:
        raise

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(TRAIN_DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
 )

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

print(f"Modelo pronto com {NUM_CLASSES} classes totais (inclui background).")
print(f"Modo de inicializacao: {model_init_mode}")
if ca_bundle is not None:
    print(f"CA bundle configurado: {ca_bundle}")
if ssl_error_message is not None:
    print(f"Erro original de SSL: {ssl_error_message}")

Modelo pronto com 7 classes totais (inclui background).
Modo de inicializacao: pretrained
CA bundle configurado: /Users/alehholiveira/Desktop/Projetos/pcb-defect-detection/.venv/lib/python3.14/site-packages/certifi/cacert.pem


In [6]:
def move_targets_to_device(targets, device):
    moved = []
    for target in targets:
        moved.append({k: v.to(device) if torch.is_tensor(v) else v for k, v in target.items()})
    return moved


def train_one_epoch(model, data_loader, optimizer, device, epoch_index):
    model.train()

    running = defaultdict(float)
    seen = 0

    pbar = tqdm(data_loader, desc=f"Treino epoca {epoch_index:02d}", leave=False)

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = move_targets_to_device(targets, device)

        loss_dict = model(images, targets)
        total_loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        batch_size = len(images)
        seen += batch_size

        running["loss"] += float(total_loss.item()) * batch_size
        for key, value in loss_dict.items():
            running[key] += float(value.item()) * batch_size

        pbar.set_postfix({"loss": f"{total_loss.item():.4f}"})

    return {key: value / max(1, seen) for key, value in running.items()}


def build_map_metric(backend="faster_coco_eval"):
    try:
        return MeanAveragePrecision(iou_type="bbox", backend=backend)
    except Exception as exc:
        print(f"Backend {backend} indisponivel ({exc}). Fallback para pycocotools.")
        return MeanAveragePrecision(iou_type="bbox")


def evaluate_map(model, data_loader, eval_device, backend="faster_coco_eval"):
    original_device = next(model.parameters()).device
    need_restore = eval_device != original_device

    if need_restore:
        model.to(eval_device)

    model.eval()
    metric = build_map_metric(backend=backend)

    with torch.no_grad():
        pbar = tqdm(data_loader, desc="Validacao mAP", leave=False)
        for images, targets in pbar:
            images_eval = [img.to(eval_device) for img in images]
            outputs = model(images_eval)

            preds = []
            refs = []

            for output, target in zip(outputs, targets):
                preds.append(
                    {
                        "boxes": output["boxes"].detach().cpu(),
                        "scores": output["scores"].detach().cpu(),
                        "labels": output["labels"].detach().cpu(),
                    }
                )

                refs.append(
                    {
                        "boxes": target["boxes"].detach().cpu(),
                        "labels": target["labels"].detach().cpu(),
                    }
                )

            metric.update(preds, refs)

    raw = metric.compute()

    model.train()
    if need_restore:
        model.to(original_device)

    scalar_keys = ["map", "map_50", "map_75", "mar_1", "mar_10", "mar_100"]
    results = {}

    for key in scalar_keys:
        if key in raw and torch.is_tensor(raw[key]) and raw[key].numel() == 1:
            results[key] = float(raw[key].item())

    return results

In [ ]:
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_dir = PROJECT_RUN_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = run_dir / "best_model.pth"
last_ckpt_path = run_dir / "last_model.pth"
history_csv_path = run_dir / "training_history.csv"
history_json_path = run_dir / "training_history.json"
history_plot_path = run_dir / "training_curves.png"
config_path = run_dir / "config.json"
class_map_path = run_dir / "class_map.json"

config_data = {
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "lr_step_size": LR_STEP_SIZE,
    "lr_gamma": LR_GAMMA,
    "ignore_category_id": IGNORE_CATEGORY_ID,
    "train_device": str(TRAIN_DEVICE),
    "eval_device": str(EVAL_DEVICE),
    "map_backend": MAP_BACKEND,
    "num_classes_total": NUM_CLASSES,
    "dataset_root": str(DATASET_ROOT),
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=2)

with open(class_map_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_to_name": {str(k): v for k, v in label_to_name.items()},
            "label_to_cat_id": {str(k): int(v) for k, v in label_to_cat_id.items()},
        },
        f,
        indent=2,
    )

print(f"Run atual: {run_dir}")

history = []
best_map = -1.0

for epoch in range(1, EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, optimizer, TRAIN_DEVICE, epoch)
    val_stats = evaluate_map(model, valid_loader, EVAL_DEVICE, backend=MAP_BACKEND)

    lr_scheduler.step()
    current_lr = float(optimizer.param_groups[0]["lr"])

    row = {
        "epoch": epoch,
        "lr": current_lr,
        **{f"train_{k}": float(v) for k, v in train_stats.items()},
        **{f"val_{k}": float(v) for k, v in val_stats.items()},
    }
    history.append(row)

    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "lr_scheduler_state_dict": lr_scheduler.state_dict(),
        "best_map": best_map,
        "config": config_data,
        "label_to_name": label_to_name,
        "label_to_cat_id": label_to_cat_id,
    }

    torch.save(checkpoint_data, last_ckpt_path)

    current_map = row.get("val_map", -1.0)
    if np.isnan(current_map):
        current_map = -1.0

    if current_map > best_map:
        best_map = current_map
        checkpoint_data["best_map"] = best_map
        torch.save(checkpoint_data, best_ckpt_path)

    print(
        f"Epoca {epoch:02d}/{EPOCHS} | "
        f"train_loss={row.get('train_loss', float('nan')):.4f} | "
        f"val_map={row.get('val_map', float('nan')):.4f} | "
        f"val_map50={row.get('val_map_50', float('nan')):.4f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(history_csv_path, index=False)
history_df.to_json(history_json_path, orient="records", indent=2)

print(f"Historico salvo em: {history_csv_path}")
print(f"Melhor checkpoint: {best_ckpt_path}")
print(f"Ultimo checkpoint: {last_ckpt_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
axes[0].set_title("Evolucao da perda de treino")
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Loss")
axes[0].legend()

if "val_map" in history_df.columns:
    axes[1].plot(history_df["epoch"], history_df["val_map"], marker="o", label="mAP")
if "val_map_50" in history_df.columns:
    axes[1].plot(history_df["epoch"], history_df["val_map_50"], marker="o", label="mAP50")

axes[1].set_title("Evolucao de mAP na validacao")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Score")
axes[1].legend()

plt.tight_layout()
plt.savefig(history_plot_path, dpi=180)
plt.show()

print(f"Curvas salvas em: {history_plot_path}")

Run atual: /Users/alehholiveira/Desktop/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/fasterrcnn_mobilenet_v3_fpn/run_20260419_185717


Treino epoca 01:   7%|▋         | 50/739 [5:39:26<79:37:50, 416.07s/it, loss=0.6811]                   

## 4. Proximos Passos
Com o treinamento concluido, execute o notebook de inferencia para aplicar o melhor checkpoint no split de teste e em imagens externas.